<div style="font-size:30px;font-weight:700;color:#111827;padding-bottom:8px;margin:18px 0;">
LangChain Expression Language (LCEL)
</div>

In [ ]:
# %%capture
# %pip install -q -U unsloth langchain-core

# GPU와 실행 환경 확인

Unsloth를 다른 Transformers 관련 라이브러리보다 먼저 가져오는 것이 좋습니다.

In [ ]:
from unsloth import FastLanguageModel
import torch

In [ ]:
print(f"PyTorch 버전: {torch.__version__}")
print(f"사용 GPU: {torch.cuda.get_device_name(0)}")

# 4비트 Qwen2.5 모델 로딩

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "unsloth/Qwen2.5-0.5B-Instruct-bnb-4bit" # 0.5B / 1.5B / 3B / 7B / 14B / 32B / 72B (7B 이상 부터 쓸만함)

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
)
model.eval()

# 커스텀 ChatModel 클래스 작성

In [ ]:
import torch
from threading import Thread
from typing import Any, Iterator, List, Optional
from pydantic import ConfigDict

In [ ]:
from langchain_core.callbacks import CallbackManagerForLLMRun
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.messages import AIMessage, AIMessageChunk, BaseMessage, HumanMessage, SystemMessage
from langchain_core.outputs import ChatGeneration, ChatGenerationChunk, ChatResult

from transformers import TextIteratorStreamer

In [ ]:
class QwenChatModel(BaseChatModel):
    """Qwen2.5-Instruct를 LangChain ChatModel로 감싸는 클래스."""

    # BaseChatModel은 Pydantic 기반이므로 필드 선언이 필요합니다.
    model: Any
    tokenizer: Any

    max_tokens: int = 512
    do_sample: bool = True
    temperature: float = 0.7
    top_p: float = 0.9

    model_config = ConfigDict(arbitrary_types_allowed=True)

    @property
    def _llm_type(self) -> str:
        return "qwen2.5-custom-chatmodel"

    def _tokenize(self, messages: List[BaseMessage]):
        """LangChain 메시지를 Qwen 채팅 형식으로 변환하고 토큰화합니다."""
        chat = []

        for message in messages:
            if isinstance(message, SystemMessage):
                role = "system"
            elif isinstance(message, HumanMessage):
                role = "user"
            elif isinstance(message, AIMessage):
                role = "assistant"
            else:
                role = "user"

            chat.append({"role": role, "content": message.content})

        inputs = self.tokenizer.apply_chat_template(
            chat,
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt",
            return_dict=True,
        )
        return inputs.to(self.model.device)

    def _generation_options(self):
        """model.generate()에 공통으로 전달할 옵션입니다."""
        options = {
            "max_length": self.max_tokens,
            "do_sample": self.do_sample,
            "pad_token_id": self.tokenizer.pad_token_id,
        }

        if self.do_sample:
            options["temperature"] = self.temperature
            options["top_p"] = self.top_p

        return options

    def _generate(
        self,
        messages: List[BaseMessage],
        stop: Optional[List[str]] = None,
        run_manager: Optional[CallbackManagerForLLMRun] = None,
        **kwargs: Any,
    ) -> ChatResult:
        """invoke()와 batch()가 사용하는 메서드입니다."""
        inputs = self._tokenize(messages)
        input_length = inputs["input_ids"].shape[1]

        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                **self._generation_options(),
            )

        new_tokens = outputs[0][input_length:]
        text = self.tokenizer.decode(
            new_tokens,
            skip_special_tokens=True,
        ).strip()

        return ChatResult(
            generations=[ChatGeneration(message=AIMessage(content=text))]
        )

    def _stream(
        self,
        messages: List[BaseMessage],
        stop: Optional[List[str]] = None,
        run_manager: Optional[CallbackManagerForLLMRun] = None,
        **kwargs: Any,
    ) -> Iterator[ChatGenerationChunk]:
        """stream()이 사용하는 메서드입니다."""
        inputs = self._tokenize(messages)

        streamer = TextIteratorStreamer(
            self.tokenizer,
            skip_prompt=True,
            skip_special_tokens=True,
        )

        thread = Thread(
            target=self.model.generate,
            kwargs={
                **inputs,
                **self._generation_options(),
                "streamer": streamer,
            },
        )
        thread.start()

        for text in streamer:
            chunk = ChatGenerationChunk(
                message=AIMessageChunk(content=text)
            )

            if run_manager:
                run_manager.on_llm_new_token(text, chunk=chunk)

            yield chunk

        thread.join()

In [ ]:
llm = QwenChatModel(model=model, tokenizer=tokenizer, max_tokens=1024)

print("ChatModel 준비 완료:", llm._llm_type)

# ChatModel 단독 실행 확인

In [ ]:
response = llm.invoke("Langchain의 LCEL이 무엇인지 한 문장으로 설명해 주세요.")

print(type(response).__name__)
print(response.content)

# LCEL의 핵심 구조

LCEL은 LangChain 구성 요소를 `|` 연산자로 연결합니다.

```python
chain = prompt | model | output_parser
```

데이터 형식은 왼쪽에서 오른쪽으로 다음과 같이 변합니다.

`dict → ChatPromptValue → AIMessage → str`

LCEL로 연결할 수 있는 객체를 LangChain에서는 **Runnable**이라고 합니다.


In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableBranch, RunnableLambda, RunnableParallel, RunnablePassthrough

In [ ]:
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "당신은 초보자에게 쉽게 설명하는 선생님입니다."),
        ("human", "{topic}에 대해 두 문장으로 설명해 주세요."),
    ]
)

output_parser = StrOutputParser()

## LCEL 없이 한 단계씩 실행

각 구성 요소의 입출력 형식을 먼저 확인하면 `|`의 의미를 쉽게 이해할 수 있습니다.


In [ ]:
prompt_value = prompt.invoke({"topic": "머신러닝"})
print("1. prompt 출력:", type(prompt_value).__name__)
print(prompt_value)

In [ ]:
ai_message = llm.invoke(prompt_value)
print("2. model 출력:", type(ai_message).__name__)
print(ai_message)

In [ ]:
text = output_parser.invoke(ai_message)
print("3. parser 출력:", type(text).__name__)
print(text)

# 같은 작업을 LCEL로 연결

왼쪽 구성 요소의 출력이 오른쪽 구성 요소의 입력으로 자동 전달됩니다.

In [ ]:
chain = prompt | llm | output_parser

In [ ]:
print(type(chain).__name__)  # RunnableSequence
print(chain.invoke({"topic": "머신러닝"}))

## 5. 모든 LCEL 체인의 공통 실행 방법

| 메서드 | 의미 |
|---|---|
| `invoke()` | 입력 하나 실행 |
| `batch()` | 입력 여러 개 실행 |
| `stream()` | 결과가 생성되는 동안 순차적으로 받기 |
| `ainvoke()` | 비동기 입력 하나 실행 |

로컬 GPU 모델의 `batch()`는 여러 스레드가 동시에 GPU를 사용하지 않도록
`max_concurrency=1`로 실행하는 것이 안전합니다.


In [ ]:
# invoke: 입력 하나
result = chain.invoke({"topic": "딥러닝"})
print(result)

In [ ]:
result = await chain.ainvoke({"topic": "딥러닝"})
print(result)

In [ ]:
# batch: 입력 여러 개
topics = [
    {"topic": "지도학습"},
    {"topic": "비지도학습"},
    {"topic": "강화학습"},
]

results = chain.batch(
    topics,
    config={"max_concurrency": 1},
)

for topic, result in zip(topics, results):
    print(f"[{topic['topic']}] {result}\n")

In [ ]:
# stream: 생성되는 텍스트를 바로 출력
for chunk in chain.stream({"topic": "트랜스포머"}):
    print(chunk, end="", flush=True)
print()

# RunnableLambda: 일반 Python 함수를 체인에 넣기

`RunnableLambda`는 Python 함수를 Runnable로 변환합니다.
입력 전처리, 출력 후처리, 계산 로직을 LCEL 체인에 넣을 때 사용합니다.


In [ ]:
def normalize_topic(text: str) -> dict:
    return {"topic": text.strip()}

normalize = RunnableLambda(normalize_topic)

In [ ]:
lambda_chain = normalize | prompt | llm | output_parser
result = lambda_chain.invoke("   인공신경망   ")

print(result)

## RunnablePassthrough: 입력을 그대로 전달하기

`RunnablePassthrough`는 입력을 바꾸지 않고 다음 단계로 전달합니다.
`assign()`을 사용하면 원래 딕셔너리를 유지하면서 새로운 키를 추가할 수 있습니다.

In [ ]:
add_question_length = RunnablePassthrough.assign(
    question_length=lambda data: len(data["question"])
)

data = add_question_length.invoke(
    {"question": "LCEL은 왜 사용하나요?"}
)

print(data)

In [ ]:
question_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "질문 길이는 {question_length}자입니다. 초보자에게 간단히 답하세요."),
        ("human", "{question}"),
    ]
)

In [ ]:
question_chain = (
    add_question_length
    | question_prompt
    | llm
    | output_parser
)

print(question_chain.invoke({"question": "LCEL은 왜 사용하나요?"}))

# RunnableParallel: 같은 입력으로 여러 작업 실행

`RunnableParallel`은 같은 입력을 여러 Runnable에 전달하고 결과를 딕셔너리로 모읍니다.

여기서는 단일 로컬 GPU에 여러 모델 호출을 동시에 보내지 않도록
가벼운 Python 함수들만 병렬 구성합니다.

In [ ]:
text_analysis = RunnableParallel(
    original=RunnablePassthrough(),
    length=RunnableLambda(len),
    upper=RunnableLambda(str.upper),
)

result = text_analysis.invoke("LangChain Expression Language")
print(result)

# 두 단계 LLM 체인 만들기

첫 번째 체인에서 키워드를 만들고, 그 결과를 두 번째 프롬프트에 전달합니다.

In [ ]:
from operator import itemgetter

In [ ]:
keyword_prompt = ChatPromptTemplate.from_template(
    "{topic}의 핵심 키워드 3개만 쉼표로 구분해 출력하세요."
)
keyword_chain = keyword_prompt | llm | output_parser

In [ ]:
summary_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "주제와 키워드를 이용해 초보자용 설명을 작성하세요."),
        ("human", "주제: {topic}\n키워드: {keywords}\n세 문장으로 설명하세요."),
    ]
)

In [ ]:
two_step_chain = (
    {
        "topic": itemgetter("topic"),
        "keywords": keyword_chain,
    }
    | summary_prompt
    | llm
    | output_parser
)

In [ ]:
result = two_step_chain.invoke({"topic": "LCEL"})
print(result)

# RunnableBranch: 조건에 따라 다른 체인 선택

`RunnableBranch`는 입력 조건에 따라 실행할 Runnable을 선택합니다.
아래 예제는 `mode`가 `short`이면 짧은 답변 체인을, 그렇지 않으면 상세 답변 체인을 실행합니다.

In [ ]:
short_prompt = ChatPromptTemplate.from_template(
    "{question}\n한 문장으로 답하세요."
)
short_chain = short_prompt | llm | output_parser

In [ ]:
detailed_prompt = ChatPromptTemplate.from_template(
    "{question}\n초보자도 이해할 수 있게 세 문장으로 답하세요."
)
detailed_chain = detailed_prompt | llm | output_parser

In [ ]:
answer_router = RunnableBranch(
    (lambda data: data["mode"] == "short", short_chain),
    detailed_chain,
)

In [ ]:
print(answer_router.invoke({"question": "Runnable이 무엇인가요?", "mode": "short"}))

## 12. 체인을 읽고 디버깅하는 방법

초보자는 다음 세 가지를 먼저 확인하면 대부분의 LCEL 오류를 찾을 수 있습니다.

1. 현재 단계의 출력 타입은 무엇인가?
2. 다음 단계가 기대하는 입력 타입은 무엇인가?
3. 프롬프트 변수 이름과 입력 딕셔너리 키가 일치하는가?


In [ ]:
print("prompt:", type(prompt).__name__)
print("llm:", type(llm).__name__)
print("output_parser:", type(output_parser).__name__)
print("chain:", type(chain).__name__)

print("prompt 입력 변수:", prompt.input_variables)